<a href="https://colab.research.google.com/github/dantae74/aice/blob/main/pre_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 필요한 라이브러리 임포트

In [ ]:
import os
import re
import joblib
import pathlib

from glob import glob
from PIL import Image

import numpy as np
import tensorflow as tf

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 2. 시험 제출을 위한 파일 이름 정의

In [ ]:
PHONE_NUMBER = "01030101825"
#MODEL_FILENAME_1 = f"{PHONE_NUMBER}_1.h5"  # Keras 모델은 .keras 또는 .h5 권장
MODEL_FILENAME_1 = f"{PHONE_NUMBER}_1.pkl"  # Scikit-learn 모델은 .pkl 또는 .joblib 권장
CSV_FILENAME_1 = f"{PHONE_NUMBER}_1.csv"

In [ ]:
#MODEL_FILENAME_2 = f"{PHONE_NUMBER}_2.h5"  # Keras 모델은 .keras 또는 .h5 권장
MODEL_FILENAME_2 = f"{PHONE_NUMBER}_2.pkl"  # Scikit-learn 모델은 .pkl 또는 .joblib 권장
CSV_FILENAME_2 = f"{PHONE_NUMBER}_2.csv"

In [ ]:
#MODEL_FILENAME_3 = f"{PHONE_NUMBER}_3.h5"  # Keras 모델은 .keras 또는 .h5 권장
MODEL_FILENAME_3 = f"{PHONE_NUMBER}_3.pkl"  # Scikit-learn 모델은 .pkl 또는 .joblib 권장
CSV_FILENAME_3 = f"{PHONE_NUMBER}_3.csv"

In [ ]:
import joblib

# 모델 저장하는 방법
# 1. tensorflow의 Keras 모델을 저장하는 방법
model.save(MODEL_FILENAME)

# 2. Scikit-learn 모델을 저장하는 방법
joblib.dump(rfc, MODEL_FILENAME)


# 3. AICE 관련 지식


## 1. 대표적인 4가지 스케일러 (StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler)

머신러닝 모델을 학습시킬 때, 데이터의 스케일(단위 및 범위)을 맞춰주는 것은 매우 중요합니다. 선형 회귀, 로지스틱 회귀, SVM, KNN, 그리고 신경망(Deep Learning) 등 거리를 계산하거나 경사하강법을 사용하는 알고리즘들은 피처의 스케일에 큰 영향을 받기 때문입니다.

대표적인 4가지 스케일러(**StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler**)의 작동 원리와 각각의 장단점을 정리해 드릴게요.

---

## 1. StandardScaler (표준화)

가장 기본적이고 널리 쓰이는 스케일러입니다. 데이터를 **평균이 0, 표준편차가 1**인 표준정규분포 형태로 변환합니다.

$$x_{scaled} = \frac{x - \mu}{\sigma}$$


*( $\mu$: 평균, $\sigma$: 표준편차 )*

* **특징:** 데이터의 최소/최대 범위가 정해지지 않습니다 (음수와 양수 모두 존재).
* **장점:** 많은 머신러닝 알고리즘(특히 선형 모델이나 PCA 같은 차원 축소 알고리즘)이 데이터가 가우시안(정규) 분포를 따른다고 가정하고 설계되었기 때문에, 이들에게 매우 잘 맞습니다.
* **단점:** **이상치(Outlier)에 매우 취약**합니다. 이상치가 존재하면 평균과 표준편차에 왜곡이 생겨, 대다수의 정상 데이터가 좁은 범위로 압착되는 현상이 발생합니다.

---

## 2. RobustScaler (로버스트 스케일링)

StandardScaler가 이상치에 취약한 점을 보완하기 위해 나온 스케일러입니다. 평균과 표준편차 대신 중앙값(Median)과 사분위수 범위(IQR: Interquartile Range)를 사용합니다.

$$x_{scaled} = \frac{x - Q_2}{Q_3 - Q_1}$$


*( $Q_1$: 25% 지점, $Q_2$: 중앙값, $Q_3$: 75% 지점 )*

* **특징:** 이름 그대로 이상치(Outliers)의 영향에 강인(Robust)합니다.
* **장점:** 데이터에 비정상적으로 크거나 작은 값(아웃라이어)이 포함되어 있어도, 중앙값과 IQR을 기준으로 정렬하기 때문에 대다수 데이터의 스케일이 균일하게 유지됩니다.
* **단점:** 표준화나 정규화에 비해 수학적으로 완전히 깔끔한 표준분포를 만들지는 않습니다.

---

## 3. MinMaxScaler (정규화)

모든 피처가 **정확히 0과 1 사이**의 값을 갖도록 스케일링합니다. (음수가 있으면 -1에서 1 사이로 조정도 가능)

$$x_{scaled} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

* **특징:** 데이터의 분포 크기(Scale)만 줄일 뿐, 원래 데이터의 분포 모양은 그대로 유지합니다.
* **장점:** 데이터의 경계(0과 1)가 명확해지므로, 이미지 처리(픽셀 값 0~255)나 K-최근접 이웃(KNN), 신경망 등 **데이터의 상대적 거리가 중요한 알고리즘**에서 뛰어난 성능을 보입니다.
* **단점:** StandardScaler와 마찬가지로 **이상치에 매우 취약**합니다. 예를 들어, 대부분의 값이 1~10 사이에 있는데 혼자 1,000인 이상치가 있다면, 정상 데이터들은 0과 0.01 사이의 아주 좁은 공간에 갇히게 됩니다.

---

## 4. MaxAbsScaler (절대최대 스케일러)

모든 값이 **-1과 1 사이**에 위치하도록 스케일링합니다. 데이터의 절대적인 최댓값이 1이 되도록 양수는 최댓값으로, 음수는 최솟값의 절대값으로 나누는 방식입니다.

$$x_{scaled} = \frac{x}{|x|_{max}}$$

* **특징:** 데이터가 이미 **0을 중심으로 공분산(Sparse)되어 있는 경우**에 주로 사용됩니다.
* **장점:** 희소 데이터(Sparse Matrix, 예: 0이 대부분인 텍스트 카운트 데이터)의 0(zero) 구조를 파괴하지 않고 스케일링을 할 수 있다는 큰 장점이 있습니다. (StandardScaler는 평균을 빼기 때문에 0이었던 값들이 0이 아니게 되어 희소성이 깨집니다.)
* **단점:** 이 역시 최댓값을 기준으로 하기 때문에 이상치에 취약합니다.

---

## 요약 및 선택 가이드

| 스케일러 | 변환 후 범위 | 핵심 기준 | 주요 특징 |
| --- | --- | --- | --- |
| **StandardScaler** | 무제한 (주로 -3 ~ 3) | 평균 = 0, 표준편차 = 1 | 회귀 분석, SVM, 대다수 알고리즘의 기본값 |
| **RobustScaler** | 무제한 | 중앙값, IQR (25%~75%) | **이상치가 많은 데이터**에 강력 추천 |
| **MinMaxScaler** | $[0, 1]$ | 최솟값, 최댓값 | 이미지, KNN, 신경망 모델, **분포를 유지해야 할 때** |
| **MaxAbsScaler** | $[-1, 1]$ | 절대 최댓값 | **희소 데이터(Sparse Data, 텍스트 등)** 보존용 |

**💡 실무 적용 팁:**
먼저 데이터에 이상치가 많은지 확인(Boxplot 등 활용)한 뒤, 이상치가 많다면 `RobustScaler`를, 데이터가 정규분포를 따르거나 일반적인 선형 모델을 쓴다면 `StandardScaler`를 기본으로 가져가는 것이 좋습니다.

어떤 스케일러를 적용할지 고민 중인 데이터의 대략적인 특성(예: 이상치 유무, 데이터의 도메인)을 알려주시면 더 알맞은 스케일러를 추천해 드릴 수 있습니다. 혹시 구상 중인 데이터셋의 형태가 어떻게 되나요?

## 사용법
파이썬의 대표적인 머신러닝 라이브러리인 Scikit-Learn(사이킷런)을 사용하면 몇 줄의 코드만으로 스케일러를 쉽게 적용할 수 있습니다.

사이킷런 스케일러를 사용할 때 가장 중요한 핵심은 **학습 데이터(`train`)로만 `fit`을 하고, 검증/테스트 데이터(`test`)는 `transform`만 해야 한다**는 점입니다. 테스트 데이터의 정보가 스케일링 과정에 스며드는 데이터 누수(Data Leakage)를 막기 위해서입니다.

아래는 가상 데이터를 만들어 각 스케일러를 적용하는 표준 예제 코드입니다.

---

## 1. 기본 사용법 (코드 예제)

```python
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler
from sklearn.model_selection import train_test_split

# 1. 가상 데이터 생성 (이상치가 포함된 데이터)
np.random.seed(42)
data = np.random.normal(loc=10, scale=2, size=(100, 2))  # 평균 10, 표준편차 2
data = np.append(data, [[100, 100], [-50, -50]], axis=0) # 극단적인 이상치 추가

df = pd.DataFrame(data, columns=['Feature_A', 'Feature_B'])

# 2. 학습용(Train) 및 테스트용(Test) 데이터 분할
X_train, X_test = train_test_split(df, test_size=0.2, random_state=42)

print("--- 원본 Train 데이터 일부 ---")
print(X_train.head(3))

```

### ① StandardScaler 사용법

```python
# 스케일러 인스턴스 생성
std_scaler = StandardScaler()

# Train 데이터로 규칙을 학습(fit)하고 변환(transform)
X_train_std = std_scaler.fit_transform(X_train)

# Test 데이터는 Train 데이터의 규칙을 '그대로' 적용하여 변환만 수행
X_test_std = std_scaler.transform(X_test)

print("\n--- StandardScaler 변환 결과 (Train) ---")
print(X_train_std[:3])

```

### ② RobustScaler 사용법

```python
robust_scaler = RobustScaler()

# 위와 동일한 메커니즘으로 fit_transform -> transform 진행
X_train_robust = robust_scaler.fit_transform(X_train)
X_test_robust = robust_scaler.transform(X_test)

print("\n--- RobustScaler 변환 결과 (Train) ---")
print(X_train_robust[:3])

```

### ③ MinMaxScaler 사용법

```python
minmax_scaler = MinMaxScaler()

X_train_minmax = minmax_scaler.fit_transform(X_train)
X_test_minmax = minmax_scaler.transform(X_test)

print("\n--- MinMaxScaler 변환 결과 (Train) ---")
print(X_train_minmax[:3])

```

---

## 2. ⚠️ 실무에서 자주 하는 실수와 주의점

### 1) `fit_transform()`은 Train 데이터에만 쓰세요

* **잘못된 예:**
```python
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)  # ❌ 절대 금지!

```


* **이유:** 테스트 데이터에 `fit_transform`을 새로 해버리면, 테스트 데이터 자체의 평균/표준편차나 최솟값/최댓값으로 스케일링이 됩니다. 이렇게 되면 학습 모델이 기준이 다른 데이터를 받아들이게 되므로 평가 결과가 왜곡됩니다.

### 2) 반환 타입은 NumPy 배열(ndarray)입니다

사이킷런의 스케일러를 거치면 판다스 데이터프레임(`DataFrame`)이 아니라 넘파이 배열(`ndarray`)로 변환됩니다. 만약 다시 데이터프레임 형태로 유지하고 싶다면 아래와 같이 재가공해야 합니다.

```python
# 다시 데이터프레임으로 만들기
X_train_scaled_df = pd.DataFrame(X_train_std, columns=X_train.columns, index=X_train.index)

```

*(참고: 최신 사이킷런 버전에서는 `scaler.set_output(transform="pandas")` 설정을 상단에 추가하면 변환 후에도 데이터프레임 형식을 유지해 줍니다.)*

### 3) 역변환 (Inverse Transform)

모델이 예측한 결과(예: 스케일링된 집값)를 사람이 읽을 수 있는 원래 단위로 되돌려야 할 때는 `inverse_transform()`을 사용합니다.

```python
# 스케일링된 데이터를 다시 원본 스케일로 복원
X_train_origin = std_scaler.inverse_transform(X_train_std)

```